# Astra: C_vote 원본 1회 + CPU 오류 채널 진단

제출용 ZIP을 만들지 않고 제출 횟수를 사용하지 않습니다. **신규 학습 없음.** GPU는 원본 C_vote의 v20 실행에만 사용합니다.

평가 데이터는 학습에 없는 64척의 이름을 실제 train 단어 대응으로 조합한 400행입니다. KoCat 표기에서 복사하지 않았지만 여전히 **합성 평가**이고 Public 성능 예측은 아닙니다.

이전 ASTRA_HIGHGAIN_KIT.zip은 사용하지 않습니다. 이번 파일은 ASTRA_CHANNEL_DIAGNOSTIC.zip입니다. 07:00 KST 이후 시작하지 않으며, 실행 중에도 07:00 또는 전체 30분에서 중단합니다. F1이 끝난 세션에서 실행하세요.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import datetime, subprocess, sys, zipfile, json, os
cutoff = datetime.datetime(2026, 10, 31, 22, tzinfo=datetime.timezone.utc)
assert datetime.datetime.now(datetime.timezone.utc) < cutoff, '07:00 KST 이후 신규 실험 금지'
drive.mount('/content/drive')
CVOTE = Path('/content/drive/MyDrive/Projects/항만문제해결을위한AI기술공모전/combo_kit/out/C_vote.zip')
assert CVOTE.is_file(), f'C_vote 경로를 실제 위치로 수정하세요: {CVOTE}'
ROOT = Path('/content/astra_channel')
ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)
print('ASTRA_CHANNEL_DIAGNOSTIC.zip 파일 하나를 업로드하세요.')
uploaded = files.upload()
assert set(uploaded) == {'ASTRA_CHANNEL_DIAGNOSTIC.zip'}, '새 진단 ZIP 하나만 업로드하세요.'
KIT = ROOT / 'kit'
assert not KIT.exists(), '이전 실행 폴더가 있습니다. 자동 덮어쓰지 않습니다.'
with zipfile.ZipFile(ROOT / 'ASTRA_CHANNEL_DIAGNOSTIC.zip') as z:
    for i in z.infolist():
        p = (KIT / i.filename).resolve()
        assert p.is_relative_to(KIT.resolve()) and ((i.external_attr >> 16) & 0o170000) != 0o120000
    z.extractall(KIT)


In [ ]:
# 기존 F1 패키지/파일은 수정하지 않습니다. pip는 현재 Colab 런타임 의존성만 확인합니다.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm==4.6.0', 'catboost==1.2.8', 'rapidfuzz==3.14.3', 'transformers==4.57.6', 'accelerate==1.9.0', 'scikit-learn==1.5.2', 'sentencepiece'], check=True)
import torch
assert torch.cuda.is_available(), '원본 v20 실행에는 CUDA GPU가 필요합니다.'
print(torch.cuda.get_device_name(0))


In [ ]:
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')
WORK = ROOT / ('run_' + stamp)
result = subprocess.run([sys.executable, str(KIT/'run_all.py'), '--cvote', str(CVOTE), '--work', str(WORK)])
receipt = WORK/'results/STATUS.json'
if receipt.exists():
    print(receipt.read_text())
archive = WORK/'ASTRA_CHANNEL_RESULTS.zip'
if archive.exists():
    files.download(str(archive))
assert result.returncode == 0, '중단/실패했습니다. 만들어진 결과 ZIP을 그대로 전달하면 됩니다.'


완료되면 **ASTRA_CHANNEL_RESULTS.zip**을 전달해 주세요. `baseline/`에 세 모델 예측과 최종 C_vote를 저장하므로 이후 CPU 분석에 GPU를 다시 쓸 필요가 없습니다. `evaluation/SUMMARY.json`은 C_vote 대비 구출/훼손, 112·HSW 공동 오답 구출, clean 피해를 구분합니다.

400행 실행 시간은 3,000행 제출 제한 통과를 뜻하지 않습니다. CPU 추가 모듈의 600회 순서·중복 검사는 전체 C_vote의 공식 600행 검사를 대체하지 않습니다. 새 제출 후보는 아직 승인된 상태가 아닙니다.
